# GodelReplay: Memory Buffer Sweep
### Does GodelPlugin's contribution grow as replay buffer shrinks?

> **Hypothesis:** The forgetting reduction delta (Replay-only - GodelReplay)
> increases as mem_size decreases, proving GodelPlugin provides complementary
> protection when replay alone is insufficient.

| mem_size | Replay-only forgetting | GodelReplay forgetting | Delta |
|----------|----------------------|----------------------|-------|
| 500 | 0.1500 (known) | 0.1487 (known) | +0.87% |
| 200 | ? | ? | ? |
| 50 | ? | ? | ? |

*Prior result (mem_size=500): GodelReplay marginally better. At smaller buffers, gap should widen.*

**Part of the Two-Layer GodelAI Architecture paper.**
*FLYWHEEL TEAM | MACP v2.3.1 | April 2026*

## 1. Install Dependencies

In [1]:
import subprocess, sys

def _install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

_install('avalanche-lib', 'torch', 'torchvision')
print('avalanche-lib installed.')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.2/134.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 993.4/993.4 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 585.2/585.2 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.5/869.5 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.4/172.4 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 534.6/534.6 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.2/165.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Load GodelAI Repository

In [2]:
import subprocess, os, sys

GODELAI_DIR = '/kaggle/working/godelai-repo'

if not os.path.exists(GODELAI_DIR):
    print('Cloning creator35lwb-web/godelai...')
    result = subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/creator35lwb-web/godelai.git', GODELAI_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError('Clone failed: ' + result.stderr)
    print('Cloned successfully.')
else:
    print('godelai-repo already present.')

if GODELAI_DIR not in sys.path:
    sys.path.insert(0, GODELAI_DIR)

from godelai.avalanche_plugin import GodelPlugin
from godelai.strategies.godel_replay import create_godel_replay_strategy
print('GodelPlugin and GodelReplay imported successfully.')


Cloning creator35lwb-web/godelai...
Cloned successfully.


2026-04-29 02:00:32.559742: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777428032.746331      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777428032.804945      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777428033.259513      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777428033.259554      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777428033.259557      23 computation_placer.cc:177] computation placer alr

GodelPlugin and GodelReplay imported successfully.


## 3. Configuration

In [3]:
import torch

def _resolve_device():
    if not torch.cuda.is_available():
        return 'cpu'
    major, minor = torch.cuda.get_device_capability(0)
    if major >= 7:
        return 'cuda'
    print('[Warning] GPU sm_' + str(major) + str(minor) + ' < sm_70 -- using CPU.')
    return 'cpu'

BASE_CONFIG = {
    'n_experiences': 10,
    'seed': 42,
    'train_epochs': 5,
    'train_mb_size': 128,
    'eval_mb_size': 256,
    'lr': 0.001,
    'device': _resolve_device(),
    'ewc_lambda': 400.0,
    'fisher_scaling': 'global_max',
    'propagation_gamma': 2.0,
    't_score_window': 50,
}
MEM_SIZES = [50, 200, 500]

print('Device: ' + BASE_CONFIG['device'])
print('Buffer sizes to test: ' + str(MEM_SIZES))
print('Strategies: replay_only, godel_replay')
print('Total runs: ' + str(len(MEM_SIZES) * 2))
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print('GPU: ' + torch.cuda.get_device_name(0) + ' | sm_' + str(cap[0]) + str(cap[1]) + ' | VRAM: ' + str(round(vram,1)) + ' GB')


[Warning] GPU sm_60 < sm_70 -- using CPU.
Device: cpu
Buffer sizes to test: [50, 200, 500]
Strategies: replay_only, godel_replay
Total runs: 6
GPU: Tesla P100-PCIE-16GB | sm_60 | VRAM: 17.1 GB


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


## 4. Run -- Buffer Size Sweep
> 6 runs total: mem_size=[50, 200, 500] x [replay_only, godel_replay]
> Estimated runtime: 90-150 min on CPU.

In [4]:
import sys
sys.path.insert(0, GODELAI_DIR)

from experiments.permutedmnist_mem_sweep import run_one, MEM_SIZES, BASE_CONFIG

results = []

for mem_size in MEM_SIZES:
    for strategy in ['replay_only', 'godel_replay']:
        r = run_one(strategy, mem_size, BASE_CONFIG)
        results.append(r)
        forg = '{:.4f}'.format(r['avg_forgetting']) if r['avg_forgetting'] is not None else 'N/A'
        acc  = '{:.4f}'.format(r['final_accuracy'])  if r['final_accuracy']  is not None else 'N/A'
        print('DONE: ' + strategy + ' mem=' + str(mem_size) + ' | forgetting=' + forg + ' acc=' + acc)

print('All runs complete.')


[Warning] GPU sm_60 < sm_70 — using CPU.

  REPLAY_ONLY | mem_size=50


100%|██████████| 9.91M/9.91M [00:00<00:00, 16.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 481kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.45MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.1MB/s]


  Task 1/10
-- >> Start of training phase << --
-- >> Start of training phase << --
100%|██████████| 469/469 [00:13<00:00, 36.00it/s]
Epoch 0 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 0.2587
	Top1_Acc_Epoch/train_phase/train_stream/Task000 = 0.9234
100%|██████████| 469/469 [00:13<00:00, 36.00it/s]
Epoch 0 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 0.2587
	Top1_Acc_Epoch/train_phase/train_stream/Task000 = 0.9234
100%|██████████| 469/469 [00:13<00:00, 35.98it/s]
Epoch 1 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 0.0980
	Top1_Acc_Epoch/train_phase/train_stream/Task000 = 0.9701
100%|██████████| 469/469 [00:13<00:00, 35.98it/s]
Epoch 1 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 0.0980
	Top1_Acc_Epoch/train_phase/train_stream/Task000 = 0.9701
100%|██████████| 469/469 [00:12<00:00, 36.29it/s]
Epoch 2 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 0.0666
	Top1_Acc_Epoch/train_phase/train_stream/Task000 = 0.9787
100%|██████████| 469/469 [00:12

/usr/local/lib/python3.12/dist-packages/avalanche/training/plugins/replay.py:123: DeprecationWarning: Call to deprecated function update (removal in version 0.7: switch to pre_adapt and post_adapt)
  self.storage_policy.update(strategy, **kwargs)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-- >> End of training phase << --
-- >> End of training phase << --
-- >> Start of eval phase << --
-- >> Start of eval phase << --
-- Starting eval on experience 0 (Task 0) from test stream --
0it [00:00, ?it/s]-- Starting eval on experience 0 (Task 0) from test stream --
100%|██████████| 40/40 [00:01<00:00, 22.06it/s]
> Eval on experience 0 (Task 0) from test stream ended.
	Loss_Exp/eval_phase/test_stream/Task000/Exp000 = 0.0689
	Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp000 = 0.9795
100%|██████████| 40/40 [00:01<00:00, 22.07it/s]
> Eval on experience 0 (Task 0) from test stream ended.
	Loss_Exp/eval_phase/test_stream/Task000/Exp000 = 0.0689
	Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp000 = 0.9795
-- Starting eval on experience 1 (Task 0) from test stream --
0it [00:00, ?it/s]-- Starting eval on experience 1 (Task 0) from test stream --
100%|██████████| 40/40 [00:01<00:00, 23.31it/s]
> Eval on experience 1 (Task 0) from test stream ended.
	Loss_Exp/eval_phase/test_stream/Tas

...

-- Starting eval on experience 0 (Task 0) from test stream --
100%|██████████| 40/40 [00:01<00:00, 21.91it/s]
> Eval on experience 0 (Task 0) from test stream ended.
	ExperienceForgetting/eval_phase/test_stream/Task000/Exp000 = 0.1944
	Loss_Exp/eval_phase/test_stream/Task000/Exp000 = 1.5142
	Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp000 = 0.7827
100%|██████████| 40/40 [00:01<00:00, 21.92it/s]
> Eval on experience 0 (Task 0) from test stream ended.
	Loss_Exp/eval_phase/test_stream/Task000/Exp000 = 1.5142
	Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp000 = 0.7827
-- Starting eval on experience 1 (Task 0) from test stream --
0it [00:00, ?it/s]-- Starting eval on experience 1 (Task 0) from test stream --
100%|██████████| 40/40 [00:01<00:00, 22.49it/s]
> Eval on experience 1 (Task 0) from test stream ended.
	ExperienceForgetting/eval_phase/test_stream/Task000/Exp001 = 0.1656
	Loss_Exp/eval_phase/test_stream/Task000/Exp001 = 1.1879
	Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp001 =

/usr/local/lib/python3.12/dist-packages/avalanche/training/plugins/replay.py:123: DeprecationWarning: Call to deprecated function update (removal in version 0.7: switch to pre_adapt and post_adapt)
  self.storage_policy.update(strategy, **kwargs)



Epoch 4 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 0.0210
	Top1_Acc_Epoch/train_phase/train_stream/Task000 = 0.9934
[GodelPlugin] Fisher Scaling: max raw=0.001438, max scaled=1.000000, scale_problem=False
[GodelPlugin] EWC-DR consolidated. Dead params: 99.9%
[GodelPlugin] Experience 10 complete. Avg T-Score: 0.9956, Sleep events: 0
-- >> End of training phase << --
-- >> End of training phase << --
-- >> Start of eval phase << --
-- >> Start of eval phase << --
-- Starting eval on experience 0 (Task 0) from test stream --
0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-- Starting eval on experience 0 (Task 0) from test stream --
100%|██████████| 40/40 [00:01<00:00, 23.01it/s]
> Eval on experience 0 (Task 0) from test stream ended.
	ExperienceForgetting/eval_phase/test_stream/Task000/Exp000 = 0.2176
	Loss_Exp/eval_phase/test_stream/Task000/Exp000 = 1.9275
	Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp000 = 0.7595
100%|██████████| 40/40 [00:01<00:00, 23.02it/s]
> Eval on experience 0 (Task 0) from test stream ended.
	Loss_Exp/eval_phase/test_stream/Task000/Exp000 = 1.9275
	Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp000 = 0.7595
-- Starting eval on experience 1 (Task 0) from test stream --
0it [00:00, ?it/s]-- Starting eval on experience 1 (Task 0) from test stream --
100%|██████████| 40/40 [00:01<00:00, 22.80it/s]
> Eval on experience 1 (Task 0) from test stream ended.
	ExperienceForgetting/eval_phase/test_stream/Task000/Exp001 = 0.2098
	Loss_Exp/eval_phase/test_stream/Task000/Exp001 = 1.6589
	Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp001 =

## 5. Results

In [5]:
print('\n' + '='*72)
print('  MEMORY BUFFER SWEEP -- PermutedMNIST (10 tasks, seed=42)')
print('='*72)
print('  ' + 'mem_size'.ljust(10) + 'Strategy'.ljust(16) +
      'Forgetting'.rjust(12) + 'Accuracy'.rjust(10) + 'Delta'.rjust(12))
print('  ' + '-'*60)

for mem_size in MEM_SIZES:
    r_replay = next((r for r in results if r['strategy'] == 'replay_only' and r['mem_size'] == mem_size), None)
    r_godel  = next((r for r in results if r['strategy'] == 'godel_replay' and r['mem_size'] == mem_size), None)
    if r_replay:
        print('  ' + str(mem_size).ljust(10) + 'replay_only'.ljust(16) +
              '{:.4f}'.format(r_replay['avg_forgetting']).rjust(12) +
              '{:.4f}'.format(r_replay['final_accuracy']).rjust(10) + '  --'.rjust(12))
    if r_godel and r_replay:
        delta = r_replay['avg_forgetting'] - r_godel['avg_forgetting']
        pct   = delta / r_replay['avg_forgetting'] * 100 if r_replay['avg_forgetting'] else 0
        sign  = '+' if pct >= 0 else ''
        print('  ' + str(mem_size).ljust(10) + 'godel_replay'.ljust(16) +
              '{:.4f}'.format(r_godel['avg_forgetting']).rjust(12) +
              '{:.4f}'.format(r_godel['final_accuracy']).rjust(10) +
              (sign + '{:.1f}%'.format(pct)).rjust(12))
        print()

print('='*72)

# Known mem_size=500 results for reference
print('\nReference (from prior run, mem_size=500):')
print('  replay_only   forgetting=0.1500  acc=0.8416')
print('  godel_replay  forgetting=0.1487  acc=0.8418  delta=+0.87%')
print('\nHypothesis: delta should INCREASE as mem_size DECREASES.')



  MEMORY BUFFER SWEEP -- PermutedMNIST (10 tasks, seed=42)
  mem_size  Strategy          Forgetting  Accuracy       Delta
  ------------------------------------------------------------
  50        replay_only           0.3902    0.6242          --
  50        godel_replay          0.4038    0.6132       -3.5%

  200       replay_only           0.2549    0.7458          --
  200       godel_replay          0.2443    0.7561       +4.1%

  500       replay_only           0.1459    0.8441          --
  500       godel_replay          0.1419    0.8478       +2.8%


Reference (from prior run, mem_size=500):
  replay_only   forgetting=0.1500  acc=0.8416
  godel_replay  forgetting=0.1487  acc=0.8418  delta=+0.87%

Hypothesis: delta should INCREASE as mem_size DECREASES.


## 6. Interpretation

**What this sweep proves:**

If delta grows as mem_size shrinks, then GodelPlugin and Replay are **complementary**:
- Replay handles distribution shift (what examples to show)
- GodelPlugin handles weight identity (which weights to protect)
- When replay budget is constrained, GodelPlugin fills the gap

**Paper implication:**
> *GodelReplay is most valuable in memory-constrained continual learning settings.
> At large buffer sizes, replay saturates protection and GodelPlugin's contribution
> is marginal. At small buffer sizes, GodelPlugin provides meaningful additional
> forgetting reduction, validating the Two-Layer Architecture complementarity claim.*

---
- [GodelAI GitHub](https://github.com/creator35lwb-web/godelai)
- [GodelAI-Lite Notebook](https://www.kaggle.com/code/creator35lwb/godelai-lite-memory-for-gemma-4)

*creator35lwb | FLYWHEEL TEAM | MACP v2.3.1*